# Train Mistral 7B Instruct LoRA (Colab Wrapper)

This notebook is a thin wrapper around `GenericLORA.py` for training `Mistral-7B-Instruct-V0.3`.

It follows the same Drive-backed workflow used elsewhere in this repository:
- mount Google Drive
- enter the synced repo root
- install dependencies
- verify the benchmark and training script exist
- optionally authenticate to Hugging Face if the model download requires it
- launch the Python training script in a fresh process

The notebook does not duplicate training logic.


In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
from pathlib import Path

# Update this to your Drive-synced repo path before running.
REPO_ROOT = Path("/content/drive/MyDrive/UVM/Deep Learning/Final Project/repo-mirror/deep-learning-final-project")
if not REPO_ROOT.exists():
    raise FileNotFoundError(f"Repo root not found: {REPO_ROOT}")

os.chdir(REPO_ROOT)
print(f"Working directory: {Path.cwd()}")

Working directory: /content/drive/MyDrive/UVM/Deep Learning/Final Project/repo-mirror/deep-learning-final-project


In [3]:
%pip install -r requirements.txt
%pip install sentence-transformers transformers datasets accelerate peft trl scikit-learn matplotlib beautifulsoup4 huggingface_hub
%pip uninstall -y torchao

print(
    "If torchao was removed or any low-level package changed, restart the runtime or rerun the setup cells before training."
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 59.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
If torchao was removed or any low-level package changed, restart the runtime or rerun the setup cells before training.


In [4]:
from pathlib import Path
import torch

from google.colab import userdata
from huggingface_hub import login

# Store your Hugging Face token in Colab Secrets with the name HF_TOKEN.
# Colab sidebar -> Secrets -> add a new secret called HF_TOKEN.
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token)
    print("Logged into Hugging Face using the HF_TOKEN Colab secret.")
else:
    print(
        "HF_TOKEN Colab secret not found. If the model is gated, add HF_TOKEN in Colab Secrets before training."
    )

required_paths = [
    Path("GenericLORA.py"),
    Path("chat_prompting.py"),
    Path("WildGraphBench"),
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required files for Llama LoRA training:\n- " + "\n- ".join(missing)
    )

for path in required_paths:
    print(f"Found: {path}")

print(f"CUDA available: {torch.cuda.is_available()}")

HF_TOKEN Colab secret not found. If the model is gated, add HF_TOKEN in Colab Secrets before training.
Found: GenericLORA.py
Found: chat_prompting.py
Found: WildGraphBench
CUDA available: True


In [5]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
DOMAIN = "culture"
TOPIC = "Marvel Cinematic Universe"
REPO_PATH = "WildGraphBench"
EMBEDDER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
OUTPUT_DIR = "Mistral-7B-Instruct-LORA-MCU"

CHUNK_SIZE = 300
TOP_K = 3
OVERLAP_THRESHOLD = 0.3
NUM_TRAIN_EPOCHS = 5
PER_DEVICE_TRAIN_BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
TRAIN_TEST_SPLIT = 0.1
LOGGING_STEPS = 10
SEED = 42
DEVICE = "cuda"
TARGET_MODULES = "q_proj,v_proj"
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.45
PREFLIGHT_MAX_LENGTH = 1024

print(f"Output checkpoint root: {OUTPUT_DIR}")

Output checkpoint root: Mistral-7B-Instruct-LORA-MCU


In [6]:
import shlex

train_cmd = [
    "python",
    "GenericLORA.py",
    "--repo_path", REPO_PATH,
    "--domain", DOMAIN,
    "--topic", TOPIC,
    "--model_name", MODEL_NAME,
    "--embedder_name", EMBEDDER_NAME,
    "--output_dir", OUTPUT_DIR,
    "--chunk_size", str(CHUNK_SIZE),
    "--top_k", str(TOP_K),
    "--overlap_threshold", str(OVERLAP_THRESHOLD),
    "--num_train_epochs", str(NUM_TRAIN_EPOCHS),
    "--per_device_train_batch_size", str(PER_DEVICE_TRAIN_BATCH_SIZE),
    "--per_device_eval_batch_size", str(PER_DEVICE_EVAL_BATCH_SIZE),
    "--gradient_accumulation_steps", str(GRADIENT_ACCUMULATION_STEPS),
    "--learning_rate", str(LEARNING_RATE),
    "--train_test_split", str(TRAIN_TEST_SPLIT),
    "--logging_steps", str(LOGGING_STEPS),
    "--seed", str(SEED),
    "--device", DEVICE,
    "--target_modules", TARGET_MODULES,
    "--lora_r", str(LORA_R),
    "--lora_alpha", str(LORA_ALPHA),
    "--lora_dropout", str(LORA_DROPOUT),
    "--preflight_max_length", str(PREFLIGHT_MAX_LENGTH),
]

train_cmd_str = " ".join(shlex.quote(part) for part in train_cmd)
print(train_cmd_str)
!{train_cmd_str}


python GenericLORA.py --repo_path WildGraphBench --domain culture --topic 'Marvel Cinematic Universe' --model_name mistralai/Mistral-7B-Instruct-v0.3 --embedder_name sentence-transformers/all-MiniLM-L6-v2 --output_dir Mistral-7B-Instruct-LORA-MCU --chunk_size 300 --top_k 3 --overlap_threshold 0.3 --num_train_epochs 5 --per_device_train_batch_size 2 --per_device_eval_batch_size 2 --gradient_accumulation_steps 4 --learning_rate 0.0002 --train_test_split 0.1 --logging_steps 10 --seed 42 --device cuda --target_modules q_proj,v_proj --lora_r 16 --lora_alpha 32 --lora_dropout 0.45 --preflight_max_length 1024
Using device: cuda
Using dtype: torch.bfloat16
config.json: 100% 601/601 [00:00<00:00, 4.80MB/s]
tokenizer_config.json: 141kB [00:00, 413MB/s]
tokenizer.json: 1.96MB [00:00, 65.6MB/s]
tokenizer.model: 100% 587k/587k [00:00<00:00, 879kB/s] 
special_tokens_map.json: 100% 414/414 [00:00<00:00, 4.44MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 23.9kB [

In [7]:
from pathlib import Path

output_path = Path(OUTPUT_DIR)
if not output_path.exists():
    raise FileNotFoundError(f"Expected output directory was not created: {output_path}")

checkpoint_dirs = sorted(output_path.glob("checkpoint-*"))
print(f"Output directory created: {output_path.resolve()}")
if checkpoint_dirs:
    print(f"Latest checkpoint: {checkpoint_dirs[-1]}")
else:
    print("Warning: no checkpoint-* directories were found yet.")

loss_curve = output_path / "loss_curve.png"
if loss_curve.exists():
    print(f"Saved loss curve: {loss_curve.resolve()}")
else:
    print("Loss curve has not been created yet.")


Output directory created: /content/drive/MyDrive/UVM/Deep Learning/Final Project/repo-mirror/deep-learning-final-project/Mistral-7B-Instruct-LORA-MCU
Latest checkpoint: Mistral-7B-Instruct-LORA-MCU/checkpoint-70
Saved loss curve: /content/drive/MyDrive/UVM/Deep Learning/Final Project/repo-mirror/deep-learning-final-project/Mistral-7B-Instruct-LORA-MCU/loss_curve.png
